# 08.02 · Durable execution and fault tolerance

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 08.01 (checkpointers, durability modes); Module 07 (graphs). |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Retry** a flaky node with `RetryPolicy`, and decide **which** errors deserve a retry.
2. **Bound** slow nodes with timeouts (`TimeoutPolicy`) and **degrade gracefully** with an `error_handler`.
3. **Resume** a crashed run from its last checkpoint, and write nodes that are safe to run twice (idempotency).
4. **Stop** a run cleanly (`RunControl.request_drain`) and continue it later.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution) | Resuming, determinism, side effects in tasks, durability modes |
| [Fault tolerance](https://docs.langchain.com/oss/python/langgraph/fault-tolerance) | `RetryPolicy`, timeouts, `error_handler`, `set_node_defaults`, draining |
| [Functional API](https://docs.langchain.com/oss/python/langgraph/functional-api) | `@task` results are checkpointed, so side effects don't repeat on resume |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langgraph-checkpoint-sqlite", "aiosqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> Everything here is 🧪 offline: the "flaky API" and the "slow search" are simulated, so failures happen exactly when we want them.

---
## Part 1 · Retries: `RetryPolicy`

In [ ]:
# 🧪 Offline: a node that fails twice with a network-style error, then works.
import operator
from typing import Annotated

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy
from typing_extensions import TypedDict


class Job(TypedDict, total=False):
    result: str
    log: Annotated[list[str], operator.add]


calls = {"n": 0}


def flaky_api(state: Job, runtime: Runtime) -> dict:
    calls["n"] += 1
    attempt = runtime.execution_info.node_attempt               # 1, 2, 3, ...
    if calls["n"] <= 2:
        raise ConnectionError(f"API unreachable (attempt {attempt})")
    return {"result": f"ok after {attempt}", "log": [f"attempt {attempt} ok"]}


def build(policy=None):
    b = StateGraph(Job)
    b.add_node("call_api", flaky_api, retry_policy=policy)
    b.add_edge(START, "call_api")
    b.add_edge("call_api", END)
    return b.compile()


fast_retry = RetryPolicy(max_attempts=3, initial_interval=0.1, backoff_factor=2.0, jitter=False,
                         retry_on=(ConnectionError, TimeoutError))
print(build(fast_retry).invoke({}))

calls["n"] = 0
try:
    build(None).invoke({})
except ConnectionError as exc:
    print("without a policy:", exc)

**You should now see** `{'result': 'ok after 3', 'log': ['attempt 3 ok']}`, then `without a policy: API unreachable (attempt 1)`.

### What just happened?

The policy re-ran the node up to `max_attempts` times, waiting `initial_interval × backoff_factor^n` (0.1 s, 0.2 s; capped by `max_interval`; `jitter=True` adds randomness so many clients don't retry in lock-step). The failed attempts wrote nothing to state: only the successful one counts. `runtime.execution_info.node_attempt` tells the node which attempt this is.

### Which errors to retry?

`RetryPolicy()` without `retry_on` uses `default_retry_on`: it retries `ConnectionError`, HTTP **5xx** errors from `requests`/`httpx` and unknown exceptions, but **not** programming errors (`ValueError`, `TypeError`, `LookupError` such as `KeyError`, `RuntimeError`, ...) and not other `OSError`s (a missing file won't appear by retrying). Check what you rely on:

In [ ]:
# 🧪 Offline: what the default policy retries (langgraph 1.2.14).
from langgraph.errors import NodeTimeoutError

from langgraph.types import default_retry_on                  # what RetryPolicy() uses when retry_on is not set

for exc in (RuntimeError("boom"), ValueError("bad input"), KeyError("x"), ConnectionError("down"),
            TimeoutError("slow"), NodeTimeoutError("search", 0.2, kind="run", run_timeout=0.2)):
    print(f"{type(exc).__name__:<17} retried: {default_retry_on(exc)}")

**You should now see** `RuntimeError`, `ValueError`, `KeyError` and `TimeoutError` → `False` (the built-in `TimeoutError` is an `OSError`), `ConnectionError` and `NodeTimeoutError` → `True`. A `RuntimeError("rate limited")` from your own code is **not** retried by default: list your transient errors in `retry_on` (a tuple of types, or a function that calls `default_retry_on` for everything else).

> The rule: **retry transient failures** (network, rate limits, 5xx, timeouts) and **never** bugs or bad input, which fail the same way every time and just waste time and tokens.

### Use the attempt number: fall back on the last try

In [ ]:
# 🧪 Offline: try the primary twice, then switch to the backup on attempt 3.
def primary_or_backup(state: Job, runtime: Runtime) -> dict:
    attempt = runtime.execution_info.node_attempt
    if attempt < 3:
        raise ConnectionError("primary model provider down")
    return {"result": "answered by the backup", "log": [f"backup on attempt {attempt}"]}


b = StateGraph(Job)
b.add_node("answer", primary_or_backup, retry_policy=fast_retry)
b.add_edge(START, "answer")
print(b.compile().invoke({}))

**You should now see** `{'result': 'answered by the backup', 'log': ['backup on attempt 3']}`. Module 04's `ModelFallbackMiddleware` does this for agents; here it's any node.

---
## Part 2 · Timeouts and error handlers

A node that **hangs** is worse than one that fails. `add_node(..., timeout=...)` cancels it after a limit and raises `NodeTimeoutError` (which the default policy retries). Timeouts work only for **`async def` nodes**: Python can cancel a coroutine, not a running thread.

In [ ]:
# 🧪 Offline: a sync node with a timeout is rejected when you compile.
import time

def slow_sync(state: Job) -> dict:
    time.sleep(1)
    return {}

try:
    StateGraph(Job).add_node("slow", slow_sync, timeout=0.1).add_edge(START, "slow").compile()
except Exception as exc:
    print(type(exc).__name__, "->", exc)

**You should now see** an error that says **Node timeouts are only supported for async nodes**. Make the node `async def` (wrap blocking calls in `await asyncio.to_thread(...)`).

In [ ]:
# 🧪 Offline: an async search that hangs; timeout + retry + error_handler.
import asyncio

from langgraph.errors import NodeError
from langgraph.types import Command, TimeoutPolicy

slow_calls = {"n": 0}


async def search(state: Job) -> dict:
    slow_calls["n"] += 1
    await asyncio.sleep(2)                                    # the API never answers in time
    return {"result": "search results", "log": ["search ok"]}


def search_failed(state: Job, error: NodeError) -> Command:
    """Runs once retries are exhausted: degrade instead of crashing the whole run."""
    return Command(update={"result": "no web results (search unavailable)",
                           "log": [f"handled {type(error.error).__name__}"]},
                   goto="write")                          # where to continue (see below)


b = StateGraph(Job)
b.add_node("search", search,
           timeout=TimeoutPolicy(run_timeout=0.2),
           retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.05, jitter=False),
           error_handler=search_failed)
b.add_node("write", lambda s: {"log": [f"write with: {s['result']}"]})
b.add_edge(START, "search")
b.add_edge("search", "write")

start = time.perf_counter()
out = await b.compile().ainvoke({})
print(out["log"])
print(f"search called {slow_calls['n']}x, total {time.perf_counter() - start:.1f}s")

**You should now see** `['handled NodeTimeoutError', 'write with: no web results (search unavailable)']` and `search called 2x, total 0.5s` (about: 2 × 0.2 s + the wait).

### What just happened?

Each attempt was cancelled at 0.2 s (`run_timeout`), the policy retried once, then the **error handler** updated state and **routed** to `write` with `Command(goto=...)`. Measured: when a handler returns a plain dict instead, the failed node's normal edges are **not** followed and the run ends after the handler. So return a `Command` whenever the graph should go on. This is the Saga idea from the docs: when a step fails for good, run a compensating step instead of failing the whole run. `TimeoutPolicy(idle_timeout=...)` is the other option: it fires only when a node stops making progress (a stream that stalls), and `runtime.heartbeat()` tells it the node is still alive.

In a cell, `await graph.ainvoke(...)` works directly (Jupyter runs an event loop). In a script use `asyncio.run(main())`.

> ⚠️ **Measured caveat (langgraph 1.2.14): error handlers and parallel siblings.** When the failing node runs in the **same super-step** as another node (a fan-out), the handler did **not** run in our test: the exception ended the run instead. Project 08's research subgraph therefore runs its two searches **one after the other** (`search_docs → search_web`). Re-test this on newer versions (exercise 3).

### Defaults for every node: `set_node_defaults`

In [ ]:
# 🧪 Offline: one retry policy for all nodes; a node's own policy still wins.
calls["n"] = 0
b = StateGraph(Job).set_node_defaults(retry_policy=fast_retry)
b.add_node("call_api", flaky_api)                   # no policy of its own
b.add_edge(START, "call_api")
print(b.compile().invoke({})["result"])

**You should now see** `ok after 3`: the default policy applied to a node that had none.

---
## Part 3 · Resume after a crash

Back to Lesson 08.01's experiment: run Project 08's graph in a child process with `durability="sync"`, kill it while it searches, then **resume it here** from the SQLite file. The graph has async nodes (the web search has a timeout), so we use `AsyncSqliteSaver`.

In [ ]:
# 🧪 Offline: crash a run in a child process (~5 s).
from mastery.graphs.research import run_and_kill

OUT = REPO_ROOT / "data" / "outputs" / "m08"
DB = OUT / "08-02-crash.sqlite"
for suffix in ("", "-wal", "-shm"):
    DB.with_name(DB.name + suffix).unlink(missing_ok=True)
QUESTION = "How do I keep the battery healthy in winter?"
print("\n".join(run_and_kill(DB, "crash-1", QUESTION, durability="sync", slow=3)))

**You should now see** the child's `PID ...`, `STEP clarify_question`, `STEP plan`, the slow search line and `💥 killed`.

In [ ]:
# 🧪 Offline: open the file in THIS process, look, then continue with input None.
from mastery.graphs.hitl import adrive, scripted_reviewer
from mastery.graphs.persistence import async_sqlite_checkpointer
from mastery.graphs.research import build_research_graph, corpus_docs_search, offline_models, offline_web_search

saver = await async_sqlite_checkpointer(DB)
agent = build_research_graph(offline_models(), web_search=offline_web_search(),
                             docs_search=corpus_docs_search(), checkpointer=saver)
cfg = {"configurable": {"thread_id": "crash-1"}}
before = await agent.aget_state(cfg)
print("next after the crash:", before.next, "| plan saved:", before.values["plan"]["queries"])

resumed = await adrive(agent, None, cfg, scripted_reviewer(["approve"]), printer=None)
print("finished:", resumed["done"], "| approval:", resumed["values"]["approval"])
print("nodes   :", [r["node"] for r in resumed["values"]["run_log"]])
await saver.conn.close()

**You should now see** `next after the crash: ('research',)` with the two saved queries, then `finished: True | approval: approved` and `nodes   : ['clarify_question', 'plan', 'search_docs', 'search_web', 'draft', 'finalize']`: `clarify_question` and `plan` ran **once** (in the dead process), research onward ran here. (`approval` isn't in `run_log`; the reviewer approved through an interrupt, Lesson 08.03.)

### What just happened?

Input `None` means "continue this thread from its newest checkpoint". LangGraph re-ran the node that was interrupted (`research`) **from its start**, not from the middle. So:

> **A node may run more than once.** Make side effects **idempotent** (sending an email twice must not send two emails: check first, or use an idempotency key), put them **after** the checkpoint that records the decision, or wrap them in a `@task` (Functional API) whose result is checkpointed. Keep nodes **deterministic** on replay: read the time or random numbers inside a task or store them in state.

---
## Part 4 · Stop cleanly: drain

Deploying a new version or shutting a worker down? `RunControl.request_drain()` asks the run to stop **at the next step boundary**: the current step finishes and is checkpointed, then the run raises `GraphDrained`. Resume later with `invoke(None, cfg)`.

In [ ]:
# 🧪 Offline: request a drain from inside step 1 (normally a signal handler or another thread does this).
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphDrained
from langgraph.runtime import RunControl

control = RunControl()


def step(name):
    def node(state: Job) -> dict:
        if name == "one":
            control.request_drain("deploying v2")
        return {"log": [name]}
    return node


b = StateGraph(Job)
for name in ("one", "two", "three"):
    b.add_node(name, step(name))
b.add_edge(START, "one")
b.add_edge("one", "two")
b.add_edge("two", "three")
drained = b.compile(checkpointer=InMemorySaver())
dcfg = {"configurable": {"thread_id": "drain-1"}}
try:
    drained.invoke({}, dcfg, control=control)
except GraphDrained as exc:
    print("drained:", exc, "| next:", drained.get_state(dcfg).next)
print("resumed:", drained.invoke(None, dcfg)["log"])

**You should now see** `drained: ...` with `next: ('two',)`, then `resumed: ['one', 'two', 'three']`. Nothing was lost and nothing ran twice.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| A `TimeoutError` or your own `RuntimeError` isn't retried | `default_retry_on` skips `OSError` (except `ConnectionError`) and `RuntimeError` | `retry_on=(ConnectionError, TimeoutError, ...)` |
| Retries burn tokens on the same bad prompt | Retrying a `ValueError`/validation error | Retry transient errors only; fix the input instead |
| "Node timeouts are only supported for async nodes" | `timeout=` on a `def` node | `async def` + `await asyncio.to_thread(blocking_call)` |
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | `asyncio.run` inside Jupyter | Use `await ...` directly in the cell |
| `NotImplementedError` from the checkpointer | `SqliteSaver` with `ainvoke` | `AsyncSqliteSaver` |
| An email/charge happened twice after a resume | The node re-ran from its start | Idempotency keys, or a `@task` that records the result |
| Run stops right after the `error_handler` | The handler returned a dict, so no edge was followed | Return `Command(update=..., goto="next_node")` |
| `error_handler` never runs | Failing node runs in parallel with another (measured, 1.2.14) | Run it in its own step, or catch inside the node |
| The child process keeps running after a notebook crash | Kernel died before `kill` | Restart the kernel; Task Manager → end `python.exe` |

## ⚠️ Production notes

> ⚠️ **Production note: budget retries.**
> 3 attempts × a 30 s timeout = 90 s worst case for one node, and every LLM attempt is billed. Set `max_attempts`, `max_interval` and timeouts per node from real latency data (LangSmith shows p95 per node).

> ⚠️ **Production note: durable doesn't mean exactly-once.**
> Checkpoints make a run resumable, but a node interrupted mid-way runs again. Anything with an external effect (payments, emails, tickets) needs an idempotency key or a check-before-act step.

## 🏋️ Exercises

1. **(Easy)** In Part 1, set `max_attempts=2`. What happens, and what does the error say?
2. **(Medium)** Change Part 2 so the search answers after 0.1 s on the **second** attempt (use `runtime.execution_info.node_attempt`). Which `log` do you get?
3. **(Medium)** Re-test the caveat: build `START → [a, b]` where `a` raises and has an `error_handler`. Does the handler run on your langgraph version? Log the version and result in `docs/research-log.md`.
4. **(Stretch)** Write a `send_email` node that is safe to re-run: it keeps a set of sent idempotency keys (a dict in a module variable is enough here) and skips a key it has already sent.

<details>
<summary><b>Show solutions</b></summary>

**1.** Attempt 1 and 2 both fail, so the run raises `ConnectionError: API unreachable (attempt 2)`.

**2.**
```python
async def search(state, runtime: Runtime):
    if runtime.execution_info.node_attempt == 1:
        await asyncio.sleep(2)
    await asyncio.sleep(0.1)
    return {"result": "search results", "log": ["search ok"]}
```
`log` becomes `['search ok', 'write with: search results']`; the handler doesn't run.

**3.** On 1.2.14 the run fails with the original exception (the handler doesn't run). If yours prints the handler's update, the behaviour changed: note it, and Project 08 could run its searches in parallel again.

**4.**
```python
SENT = set()
def send_email(state):
    key = f"{state['thread_id']}:{state['draft_hash']}"
    if key in SENT:
        return {"log": ["email already sent"]}
    SENT.add(key)          # real code: the email provider's idempotency key, or a DB unique constraint
    return {"log": ["email sent"]}
```
</details>

## ✅ Checkpoint

- [ ] I added a `RetryPolicy` with an explicit `retry_on` and can say why `ValueError` isn't in it.
- [ ] I bounded an async node with a timeout and degraded with an `error_handler`.
- [ ] I resumed a killed run with `invoke(None, cfg)` and know a node can run twice.
- [ ] I drained a run and resumed it.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Fault tolerance](https://docs.langchain.com/oss/python/langgraph/fault-tolerance) and note in [`docs/research-log.md`](../../docs/research-log.md): the difference between `run_timeout` and `idle_timeout`, what `refresh_on` does, and whether the docs mention the error-handler behaviour with parallel nodes.

## Summary

- `RetryPolicy` for transient errors (list them in `retry_on`), never for bugs.
- Timeouts need `async def` nodes; `error_handler` turns a final failure into a normal update.
- Resume = `invoke(None, cfg)` from the newest checkpoint; the interrupted node re-runs from its start, so make effects idempotent.
- `RunControl.request_drain()` stops at a step boundary; resume the same way.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.03 · Interrupts and human-in-the-loop (graph level)** → [`08-03-interrupts.ipynb`](08-03-interrupts.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)